In [ ]:
!pip install sentence-transformers

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch

In [ ]:
df_ing = pd.read_csv("final_dataset.csv")
df_cal = pd.read_csv("final_cleaned_data.csv")

In [ ]:
df_ing.rename(columns={"meal_name": "name"}, inplace=True)

df_ing["name"] = df_ing["name"].str.lower().str.strip()
df_cal["name"] = df_cal["name"].str.lower().str.strip()

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
df_ing["name"] = df_ing["name"].fillna("").astype(str)
df_cal["name"] = df_cal["name"].fillna("").astype(str)

In [ ]:
ing_names = df_ing["name"].tolist()
cal_names = df_cal["name"].tolist()

emb_ing = model.encode(ing_names, convert_to_tensor=True)
emb_cal = model.encode(cal_names, convert_to_tensor=True)

In [ ]:
similarity = util.cos_sim(emb_ing, emb_cal)

In [ ]:
best_matches = similarity.argmax(dim=1)

df_ing["matched_name"] = [cal_names[i.item()] for i in best_matches]

In [ ]:
merged = pd.merge(df_ing, df_cal, left_on="matched_name", right_on="name", how="left")

In [ ]:
# 1) إعادة تسمية الأعمدة
merged = merged.rename(columns={
    "name_x": "name",
    "matched_name_x": "matched_name",
    "carbs": "carb",      # لو عندك carbs
    "protien": "protein"  # لو كان مكتوب غلط
})

# 2) حذف الأعمدة اللي مش عايزينها (لو موجودة)
cols_to_drop = ["name_y", "matched_name_y"]
merged = merged.drop(columns=[c for c in cols_to_drop if c in merged.columns])

# 3) ترتيب الأعمدة النهائي
final_cols = [
    "name",
    "matched_name",
    "ingredients",
    "category",
    "portion",
    "calories",
    "protein",
    "fat",
    "carb"
]

merged = merged[final_cols]

# 4) حفظ الملف
merged.to_csv("final_matched_dataset.csv", index=False)